# 03 · EDA step 2: sanity checks

Before any analysis, confirm the 2025-26 data is complete, consistent, and that the target is
defined the way the plan says.

1. **Results mix:** regulation, overtime and shootout games, and how many ties get dropped
2. **Consistency:** both team rows of a game agree, and the counts make sense
3. **Coverage:** games per team, and games per month (how the season is spread across the folds)
4. **Missing values:** in the raw counts and in the pre-game features after the warm-up

Every check is an `assert` where there's a clear right answer. The rest are printed counts
to read. Only summaries are printed, never game-level rows.

In [ ]:
%run ./01_pregame_features.ipynb

In [ ]:
GAMES_SQL = """
SELECT g.game_id, g.game_date, g.home_team_id, g.away_team_id, g.home_goals, g.away_goals,
       g.is_neutral, g.ot_periods, g.shootout_winner_team_id, g.game_type
FROM hockey.games g
WHERE g.season = %s AND g.is_played
"""
TEAM_EXTRA_SQL = """
SELECT game_id, team_id, ot_periods, shootout_won, en_goals, sh_goals, pim, penalties, rest_days
FROM features.team_games WHERE season = %s
"""

def query(conn, sql, params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


with psycopg.connect(settings.database_url) as conn:
    games = query(conn, GAMES_SQL, (SEASON,))
    extra = query(conn, TEAM_EXTRA_SQL, (SEASON,))
# Same games the features use: both teams D-I (see 01)
games = games[games["game_id"].isin(team_games["game_id"])].reset_index(drop=True)
new_cols = ["game_id", "team_id"] + [c for c in extra.columns if c not in team_games.columns]
tg = team_games.merge(extra[new_cols], on=["game_id", "team_id"], how="left", validate="one_to_one")
print(f"{len(games)} played games, {len(tg)} team-game rows, {tg['team_id'].nunique()} teams")

## 1. Results mix

In [ ]:
margin = (games["home_goals"] - games["away_goals"]).abs()
kind = pd.Series("regulation", index=games.index, name="ended in")
kind[games["ot_periods"] > 0] = "overtime"
kind[games["shootout_winner_team_id"].notna()] = "shootout"
outcome = pd.Series("decided", index=games.index, name="result")
outcome[margin == 0] = "tie"

mix = pd.crosstab(kind, outcome, margins=True, margins_name="total")
mix

In [ ]:
# Shootouts are recorded as ties (the NCAA counts them as ties), so they're dropped with the ties.
assert (margin[games["shootout_winner_team_id"].notna()] == 0).all(), "shootout game with a winner on goals"
# A regulation game can't end level, and overtime games are decided by one goal unless tied
assert (margin[kind == "regulation"] > 0).all(), "tie without overtime"
assert margin[(kind == "overtime")].isin([0, 1]).all(), "overtime game decided by more than 1"

ties = int((margin == 0).sum())
print(f"Ties dropped from the target: {ties} of {len(games)} ({ties / len(games):.1%})")
print(f"  of which settled by shootout: {int((kind == 'shootout').sum())}")
print(f"Overtime winners kept as wins/losses: {int(((kind == 'overtime') & (margin > 0)).sum())}")
print(f"Game types: {games['game_type'].fillna('none').value_counts().to_dict()}")

## 2. Consistency between the two team rows of each game

In [ ]:
per_game = tg.groupby("game_id")
assert (per_game.size() == 2).all(), "a game without exactly two team rows"
assert (per_game["is_home"].sum() == 1).all(), "a game without exactly one home row"
assert (per_game["team_id"].nunique() == 2).all(), "a team playing itself"

# Goals, shots and special teams mirror each other
both = tg.merge(tg, left_on=["game_id", "opponent_team_id"], right_on=["game_id", "team_id"], suffixes=("", "_opp"))
for mine, theirs in [("goals_for", "goals_against_opp"), ("shots_for", "shots_against_opp"),
                     ("pp_goals", "pp_goals_against_opp"), ("pp_opportunities", "times_shorthanded_opp")]:
    bad = (both[mine] != both[theirs]).sum()
    print(f"{mine:>16} vs opponent's {theirs.removesuffix('_opp'):<18} mismatches: {bad}")

# The games table and the team rows agree on the score
home_rows = tg[tg["is_home"]].set_index("game_id")
g = games.set_index("game_id")
assert (home_rows.loc[g.index, "goals_for"].values == g["home_goals"].values).all()
assert (home_rows.loc[g.index, "goals_against"].values == g["away_goals"].values).all()

# Counts that can't be negative or exceed their parts
assert (tg[["goals_for", "goals_against", "shots_for", "shots_against", "pp_goals", "pp_opportunities"]] >= 0).all().all()
print(f"PP goals > PP chances: {(tg['pp_goals'] > tg['pp_opportunities']).sum()} rows")
assert (tg["goals_for"] <= tg["shots_for"]).all(), "more goals than shots (non-D-I data leaking in?)"
print("Goals never exceed shots on goal.")
print(f"Empty-net goals > goals: {(tg['en_goals'] > tg['goals_for']).sum()} rows")

## 3. Coverage

In [ ]:
gp = tg.groupby("team_id").size()
print(f"Games per team: min {gp.min()}, median {int(gp.median())}, max {gp.max()} ({len(gp)} teams)")
print(f"Teams with fewer than 20 games: {(gp < 20).sum()}")

by_month = games.assign(month=pd.to_datetime(games["game_date"]).dt.to_period("M")).groupby("month").size()
by_month.rename("games").to_frame().T

In [ ]:
# Where the locked k = 5 folds fall in the season (games after warm-up and dropping ties)
from sklearn.model_selection import TimeSeriesSplit

K = 5
spans = []
for i, (train, test) in enumerate(TimeSeriesSplit(n_splits=K).split(rows), start=1):
    d = rows["game_date"].iloc[test]
    spans.append({"fold": i, "train games": len(train), "test games": len(test),
                  "test from": d.min().date(), "test to": d.max().date()})
pd.DataFrame(spans).set_index("fold")

In [ ]:
rest = tg["rest_days"].dropna()
print(f"Rest days: median {rest.median():.0f}, 1 day (back-to-back) {(rest == 1).mean():.0%}, "
      f"0 days {(rest == 0).sum()} rows, over 14 days {(rest > 14).sum()} rows")
print(f"Neutral-site games: {int(games['is_neutral'].sum())}")
print(f"Home win rate (decided games): {rows['home_win'].mean():.3f} after warm-up, "
      f"{table['home_win'].mean():.3f} all games")

## 4. Missing values

In [ ]:
raw_cols = ["goals_for", "goals_against", "shots_for", "shots_against", "pp_goals",
            "pp_opportunities", "pp_goals_against", "times_shorthanded", "en_goals", "pim"]
raw_missing = tg[raw_cols].isna().sum()
print(f"Missing raw counts: {int(raw_missing.sum())}")
assert raw_missing.sum() == 0

diff_cols = [c for c in rows.columns if c.startswith("diff_")]
miss = rows[diff_cols].isna().sum()
print(f"Pre-game features after warm-up: {len(diff_cols)} columns, "
      f"{int((miss > 0).sum())} with any missing value")
miss[miss > 0].rename("missing games").to_frame() if (miss > 0).any() else None

## Findings

**The data passes. One fix was needed, and it's now in `01`.**

- **Fix: non-D-I games removed.** One game against a D-III team had 0 shots recorded for
  that team, which scored 3 goals. CHN records non-D-I teams only partly. The game was
  already dropped from the target by the warm-up, but it still counted in the D-I
  opponent's season stats as 52–0 shots, distorting that team's shot share and save %. `01`
  now loads only D-I vs D-I games: 1,147 games and 63 teams. Rerunning step 1 changed
  nothing: the ranking is still identical at k = 3–5, and k = 5 stays locked.
- **Results mix:**
  - 938 regulation games, 153 overtime games (130 decided, 23 ties) and 56 shootouts.
  - Shootouts count as ties, so 79 games (6.9%) drop from the target. Overtime winners
    count as wins.
- **Consistency:**
  - Both team rows of every game mirror each other: goals, shots, PP goals and PP
    chances, with zero mismatches.
  - The scores match the games table.
  - There are no impossible counts: PP goals never exceed PP chances, goals never exceed
    shots, and empty-net goals never exceed goals.
- **Coverage:**
  - Every team played 30–43 games (median 36).
  - Games per month: about 100 in December (holiday break) and March, and about 200–270 in
    the other months.
- **The k = 5 folds:** each test fold has 159 games, covering about three weeks each from
  mid-November to late February. The last fold runs from Feb 21 to Apr 11 and holds the
  conference and NCAA tournaments, so report it separately as planned.
- **Missing values:** none in the raw counts, and none in the 24 pre-game features after the
  warm-up.
- **Context worth knowing for later steps:**
  - Home teams win 55.4% of decided games after the warm-up. That's the baseline any
    model has to beat.
  - 58 games are at neutral sites.
  - 44% of team-games come the day after that team's previous game (mostly Friday/Saturday
    series), which makes the back-to-back feature worth testing in step 6.
  - The median rest is 4 days.